# 1.1 Introduction to Systems of Linear Equations

**Course:** Elementary Linear Algebra (Larson, 8e) 
**Chapter 1:** Systems of Linear Equations

---

### What you will learn

- What a **system of linear equations** is and why it matters.
- How to represent a system as an **augmented matrix**.
- The three **elementary row operations** (swap, scale, add-scaled-row).
- **Forward elimination** to reach row-echelon form (REF).
- **Back-substitution** to extract the solution.
- The three possible outcomes: **unique**, **infinitely many**, or **no** solution.
- How to handle **parametric (free-variable)** solutions.

### Prerequisites

- Basic algebra (solving two equations in two unknowns by substitution/elimination).
- Python fundamentals (lists, loops, functions).

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.elimination import (
    swap_rows,
    scale_row,
    add_scaled_row,
    forward_eliminate,
    back_substitute,
    solve_system,
)
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

print("Setup complete.")

---

## 2. Motivation --- Chemical Equation Balancing

Consider the combustion of propane:

$$x_1 \text{C}_3\text{H}_8 + x_2 \text{O}_2 \longrightarrow x_3 \text{CO}_2 + x_4 \text{H}_2\text{O}$$

Conservation of atoms gives us one equation per element:

| Element  | Left side          | Right side         | Equation                     |
|----------|--------------------|--------------------|------------------------------|
| Carbon   | $3x_1$             | $x_3$              | $3x_1 = x_3$                |
| Hydrogen | $8x_1$             | $2x_4$             | $8x_1 = 2x_4$               |
| Oxygen   | $2x_2$             | $2x_3 + x_4$       | $2x_2 = 2x_3 + x_4$         |

Moving everything to the left side:

$$\begin{aligned}
3x_1 + 0x_2 - x_3 + 0x_4 &= 0 \\
8x_1 + 0x_2 + 0x_3 - 2x_4 &= 0 \\
0x_1 + 2x_2 - 2x_3 - x_4 &= 0
\end{aligned}$$

This is a **homogeneous system** of 3 equations in 4 unknowns. We have more unknowns
than equations, so we expect a family of solutions rather than a single answer.

> **Question:** Can we find $x_1, x_2, x_3, x_4$ *systematically* --- without guesswork?

In [ ]:
# Encode the system as an augmented matrix [A | b]
# Each row is one equation; the last column is the right-hand side (all zeros here).
chem_data = [
    [3,  0, -1,  0, 0],   # Carbon:   3x1 - x3 = 0
    [8,  0,  0, -2, 0],   # Hydrogen: 8x1 - 2x4 = 0
    [0,  2, -2, -1, 0],   # Oxygen:   2x2 - 2x3 - x4 = 0
]

chem = Matrix.from_lists(chem_data)
print("Augmented matrix for the chemical system:")
print(chem)

We will revisit this system throughout the notebook. By the end, you will be able to
solve it completely and recover the balanced equation
$\text{C}_3\text{H}_8 + 5\text{O}_2 \to 3\text{CO}_2 + 4\text{H}_2\text{O}$.

---

## 3. Build --- Core Concepts

We will build up the machinery piece by piece:

1. Representing a system as a list of lists (then as a `Matrix`)
2. Definition of a linear equation
3. Row swap
4. Row scaling
5. Adding a scaled row
6. Row-echelon form
7. Forward elimination
8. Back-substitution
9. Consistent vs. inconsistent systems
10. Parametric solutions and free variables

### 3.1 Representing a system as a list of lists

Every system of linear equations can be written in the form $A\mathbf{x} = \mathbf{b}$,
where $A$ is the **coefficient matrix**, $\mathbf{x}$ is the vector of unknowns, and
$\mathbf{b}$ is the right-hand side. The **augmented matrix** $[A \mid \mathbf{b}]$
packs all of this information into a single rectangular array.

For example, the system

$$\begin{cases}
 x + y + 2z = 9 \\
 2x + 4y - 3z = 1 \\
 3x + 6y - 5z = 0
\end{cases}$$

becomes the augmented matrix

$$\left[\begin{array}{ccc|c}
1 & 1 & 2 & 9 \\
2 & 4 & -3 & 1 \\
3 & 6 & -5 & 0
\end{array}\right]$$

In [ ]:
# Step 1: Start with a plain list of lists (one list per equation).
aug_data = [
    [1, 1,  2, 9],
    [2, 4, -3, 1],
    [3, 6, -5, 0],
]

# Step 2: Wrap it in our Matrix class for structured access.
aug = Matrix.from_lists(aug_data)
print("Augmented matrix:")
print(aug)

# Access individual entries using (row, col) indexing (0-based).
print(f"\nEntry at row 0, col 2: {aug[0, 2]}")
print(f"Full row 1: {aug.get_row(1)}")
print(f"Full col 3 (right-hand side): {aug.get_col(3)}")

### 3.2 Definition: Linear equation in $n$ variables

> **Definition (Larson, Sec. 1.1).** A **linear equation in $n$ variables** $x_1, x_2, \ldots, x_n$
> has the form
>
> $$a_1 x_1 + a_2 x_2 + \cdots + a_n x_n = b$$
>
> where the coefficients $a_1, a_2, \ldots, a_n$ and the constant $b$ are real numbers.

Key properties:

- Each variable appears to the **first power** (no $x^2$, no $xy$, no $\sin x$).
- A **system** of linear equations is a collection of one or more such equations
  involving the same variables.
- A **solution** is an assignment of values to $x_1, \ldots, x_n$ that satisfies *every*
  equation simultaneously.

In [ ]:
# Quick check: is a given equation linear?
# Linear equations look like a1*x1 + a2*x2 + ... + an*xn = b
# The coefficients are just numbers --- no products of variables, no powers > 1.

examples = [
    ("3x + 2y - z = 7",        True,  "all variables to first power"),
    ("x^2 + y = 4",            False, "x appears squared"),
    ("xy + z = 1",             False, "product of variables"),
    ("5x1 - 3x2 + 0x3 = 10",  True,  "zero coefficient is fine"),
    ("sin(x) + y = 2",         False, "nonlinear function of x"),
]

print(f"{'Equation':<25} {'Linear?':<10} {'Reason'}")
print("-" * 65)
for eq, is_lin, reason in examples:
    print(f"{eq:<25} {str(is_lin):<10} {reason}")

### 3.3 Elementary row operation: Swap rows

The first of three **elementary row operations** is swapping two rows:

$$R_i \leftrightarrow R_j$$

This corresponds to rewriting the equations in a different order --- it
does not change the solution set.

In [ ]:
# Work on a copy so we can reuse the original.
m = chem.copy()
print("Before swap:")
print(m)

# Swap row 0 and row 2 (R1 <-> R3 in math notation).
swap_rows(m, 0, 2)
print("\nAfter R1 <-> R3:")
print(m)

### 3.4 Elementary row operation: Scale a row

The second operation multiplies every entry in a row by a nonzero scalar:

$$R_i \leftarrow c \cdot R_i, \quad c \neq 0$$

This is equivalent to multiplying both sides of an equation by the same constant.

In [ ]:
# Start fresh from the chemical system.
m = chem.copy()
print("Before scaling:")
print(m)

# Divide row 0 by 3 (i.e., scale by 1/3) to make the leading entry 1.
scale_row(m, 0, 1/3)
print("\nAfter R1 <- (1/3) * R1:")
print(m)

### 3.5 Elementary row operation: Add a scaled row to another

The third --- and most powerful --- operation adds a multiple of one row to another:

$$R_i \leftarrow R_i + c \cdot R_j$$

This is the workhorse of elimination: we use it to create zeros below
(and later above) pivot positions.

In [ ]:
# Continue from the scaled matrix.
print("Current state (after scaling R1):")
print(m)

# Eliminate the 8 in position (1, 0) by subtracting 8 times row 0.
# That is: R2 <- R2 + (-8) * R1
add_scaled_row(m, 1, 0, -8)
print("\nAfter R2 <- R2 + (-8)*R1:")
print(m)
print("\nNotice: the entry in position (1,0) is now 0.")

### 3.6 Definition: Row-echelon form (REF)

> **Definition (Larson, Sec. 1.2).** A matrix is in **row-echelon form** if it satisfies
> all of the following:
>
> 1. All rows consisting entirely of zeros are at the **bottom**.
> 2. The first nonzero entry in each nonzero row is a **1** (called the **leading 1** or **pivot**).
> 3. Each leading 1 is to the **right** of the leading 1 in the row above it (staircase pattern).
> 4. All entries in a column **below** a leading 1 are zeros.

Schematically (with `*` denoting any value):

$$\begin{bmatrix}
1 & * & * & * \\
0 & 1 & * & * \\
0 & 0 & 1 & * \\
0 & 0 & 0 & 0
\end{bmatrix}$$

Once a matrix is in REF, the solution can be read off by **back-substitution** (starting
from the bottom row and working upward).

In [ ]:
# Example: check whether given matrices are in REF.

ref_yes = Matrix.from_lists([
    [1, 3, 0, 2],
    [0, 1, 5, 1],
    [0, 0, 1, 4],
])

ref_no = Matrix.from_lists([
    [1, 3, 0, 2],
    [0, 0, 5, 1],   # leading entry is 5, not 1
    [0, 1, 1, 4],   # leading 1 is to the LEFT of the row above
])

print("Matrix A (is in REF):")
print(ref_yes)
print("\nMatrix B (NOT in REF):")
print(ref_no)
print("\nMatrix B violates conditions 2 and 3:")
print("  - Row 2 has leading entry 5 (should be 1).")
print("  - Row 3's leading 1 is in column 2, but row 2's first nonzero is in column 3.")

### 3.7 Forward elimination (Gaussian elimination)

**Forward elimination** systematically applies the three row operations to convert
any matrix into row-echelon form. The algorithm:

1. Start at the top-left corner.
2. **Partial pivoting**: swap rows so the entry with the largest absolute value is
   in the pivot position (improves numerical stability).
3. **Scale** the pivot row so the leading entry becomes 1.
4. **Eliminate** all entries below the pivot by adding appropriate multiples.
5. Move to the next pivot position (one row down, one or more columns right) and repeat.

Let us apply this to the chemical equation system and watch each step.

In [ ]:
# Fresh copy of the chemical system.
m = chem.copy()
print("Original augmented matrix:")
print(m)
print("\n" + "=" * 50)
print("Forward elimination (verbose):")
print("=" * 50 + "\n")

swaps = forward_eliminate(m, verbose=True)

print("\n" + "=" * 50)
print(f"Row-echelon form reached ({swaps} row swap(s)):")
print(m)

### 3.8 Back-substitution

Once the matrix is in REF, we solve from the **bottom up**:

1. The last nonzero row gives the value of one variable directly.
2. Substitute upward into each preceding row to find the remaining variables.

Our `back_substitute` function returns a tuple `(solution_type, result)` where:

- `"unique"` --- a single solution vector.
- `"infinite"` --- a parametric family (free variables become parameters).
- `"inconsistent"` --- no solution exists.

In [ ]:
# m is already in REF from the previous cell.
sol_type, result = back_substitute(m)

print(f"Solution type: {sol_type}")
print()

if sol_type == "unique":
    print(f"Solution: {result}")
elif sol_type == "infinite":
    print("Free variables:", result["free_vars"])
    print("Parameter names:", result["parametric_names"])
    print("Particular solution:", result["particular"])
    print("\nCoefficients for each free variable:")
    for fv, coeffs in result["coefficients"].items():
        name = result["parametric_names"][fv]
        print(f"  {name} (x{fv+1}): {coeffs}")
else:
    print("The system is inconsistent (no solution).")

### 3.9 Definition: Consistent vs. Inconsistent Systems

> **Definition.** A system of linear equations is **consistent** if it has at least one
> solution, and **inconsistent** if it has no solution.

There are exactly **three possibilities** for a system of linear equations:

| Outcome              | What it looks like in REF                                    |
|----------------------|--------------------------------------------------------------|
| **Unique solution**  | Every variable is a pivot variable (no free variables).      |
| **Infinitely many**  | At least one free variable; no contradictory row.            |
| **No solution**      | A row of the form $[0 \ 0 \ \cdots \ 0 \mid c]$ with $c \neq 0$. |

This trichotomy is a **fundamental theorem** --- there is no fourth possibility.

In [ ]:
# Demonstrate all three cases.

# Case 1: Unique solution
A1 = Matrix.from_lists([[1, 1, 2, 9],
                         [2, 4, -3, 1],
                         [3, 6, -5, 0]])
t1, r1 = solve_system(A1)
print(f"Case 1 --- {t1}: {r1}")

# Case 2: Infinitely many solutions (more unknowns than equations)
A2 = Matrix.from_lists([[1, 2, 3],
                         [2, 4, 6]])
t2, r2 = solve_system(A2)
print(f"\nCase 2 --- {t2}")
if t2 == "infinite":
    print(f"  Free vars: {r2['free_vars']}")

# Case 3: Inconsistent (contradiction)
A3 = Matrix.from_lists([[1, 1, 1, 2],
                         [0, 1, 1, 1],
                         [0, 0, 0, 3]])  # 0 = 3 is impossible
t3, r3 = solve_system(A3)
print(f"\nCase 3 --- {t3}: {r3}")

### 3.10 Parametric solutions and free variables

When a system has **more unknowns than pivot columns**, the non-pivot variables are
called **free variables**. We assign them arbitrary parameters ($t$, $s$, etc.) and
express the pivot variables in terms of those parameters.

**Back to the chemical equation.** The system has 4 unknowns but only 3 equations
(and 3 pivots), so one variable is free. The back-substitution result tells us
which variable is free and how the others depend on it.

To get integer coefficients for the balanced equation, we pick a convenient value
for the free parameter.

In [ ]:
# Solve the chemical system from scratch.
chem_fresh = Matrix.from_lists([
    [3,  0, -1,  0, 0],
    [8,  0,  0, -2, 0],
    [0,  2, -2, -1, 0],
])

sol_type, result = solve_system(chem_fresh)
print(f"Solution type: {sol_type}\n")

if sol_type == "infinite":
    fv = result["free_vars"]
    particular = result["particular"]
    coefficients = result["coefficients"]
    names = result["parametric_names"]

    print("General solution:")
    for i in range(4):
        terms = [f"{particular[i]:.4f}"]
        for f_idx in fv:
            c = coefficients[f_idx][i]
            if abs(c) > EPSILON:
                terms.append(f"{c:+.4f}*{names[f_idx]}")
        print(f"  x{i+1} = {' '.join(terms)}")

    # Choose the free parameter to get the standard balanced equation.
    # Set x4 = t1 = 4 (smallest positive integer that gives all-integer coefficients).
    t_val = 4
    print(f"\nSetting {names[fv[0]]} = {t_val}:")
    solution = [0.0] * 4
    for i in range(4):
        solution[i] = particular[i]
        for f_idx in fv:
            solution[i] += coefficients[f_idx][i] * t_val
    
    x1, x2, x3, x4 = [int(round(v)) for v in solution]
    print(f"  x1={x1}, x2={x2}, x3={x3}, x4={x4}")
    print(f"\nBalanced equation: {x1}C3H8 + {x2}O2 -> {x3}CO2 + {x4}H2O")

### Full implementation with `solve_system`

The `solve_system` function wraps `forward_eliminate` and `back_substitute` into
a single call. It makes a copy of the input, so the original matrix is not modified.

In [ ]:
# Solve the 3x3 system from section 3.1 using the one-line interface.
system = Matrix.from_lists([
    [1, 1,  2, 9],
    [2, 4, -3, 1],
    [3, 6, -5, 0],
])

sol_type, result = solve_system(system)
print(f"Solution type: {sol_type}")

if sol_type == "unique":
    for i, val in enumerate(result):
        print(f"  x{i+1} = {val:.6f}")

# Verify: the original matrix should be untouched.
print(f"\nOriginal matrix unchanged: {system[0, 0] == 1.0 and system[1, 0] == 2.0}")

---

## 4. Verify --- Cross-check with NumPy

Our from-scratch implementation should agree with NumPy's highly optimized solvers.
We test all three solution types.

In [ ]:
# --- Test 1: Unique solution ---
A_data = [[1, 1, 2], [2, 4, -3], [3, 6, -5]]
b_data = [9, 1, 0]

# Our solver
aug_test = Matrix.from_lists([[1, 1, 2, 9],
                               [2, 4, -3, 1],
                               [3, 6, -5, 0]])
our_type, our_sol = solve_system(aug_test)

# NumPy
np_A = np.array(A_data, dtype=float)
np_b = np.array(b_data, dtype=float)
np_sol = np.linalg.solve(np_A, np_b)

print("Test 1: Unique solution")
print(f"  Our solution:   {[round(v, 6) for v in our_sol]}")
print(f"  NumPy solution: {[round(v, 6) for v in np_sol]}")

# Check residual: plug our solution back into A*x - b.
residual = np_A @ np.array(our_sol) - np_b
residual_norm = np.linalg.norm(residual)
print(f"  Residual norm:  {residual_norm:.2e}")
assert residual_norm < EPSILON, "Residual too large!"
print("  PASSED")

In [ ]:
# --- Test 2: Inconsistent system ---
aug_incon = Matrix.from_lists([
    [1, 1, 1, 2],
    [0, 1, 1, 1],
    [0, 0, 0, 3],   # This row says 0 = 3, which is impossible.
])

inc_type, inc_result = solve_system(aug_incon)
print("Test 2: Inconsistent system")
print(f"  Solution type: {inc_type}")
print(f"  Result: {inc_result}")
assert inc_type == "inconsistent", "Should be inconsistent!"
print("  PASSED")

In [ ]:
# --- Test 3: Underdetermined system (infinitely many solutions) ---
aug_under = Matrix.from_lists([
    [1, 2, 3],
    [2, 4, 6],   # Row 2 is exactly 2 * Row 1.
])

und_type, und_result = solve_system(aug_under)
print("Test 3: Underdetermined system")
print(f"  Solution type: {und_type}")
assert und_type == "infinite", "Should have infinitely many solutions!"
print(f"  Free variables: {und_result['free_vars']}")
print(f"  Particular: {und_result['particular']}")
print("  PASSED")

---

## 5. Visualize --- Geometry of Linear Systems

### 5.1 Two dimensions: lines in the plane

In 2D, each linear equation $ax + by = c$ represents a **line**. A system of two
equations corresponds to two lines, and the solution is their intersection.

| Geometry                    | Algebraic outcome       |
|-----------------------------|-------------------------|
| Two lines cross at a point  | Unique solution         |
| Two lines are parallel      | No solution (inconsistent) |
| Two lines are the same      | Infinitely many solutions |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.linspace(-2, 6, 300)

# --- Case 1: Unique solution (intersecting lines) ---
# x + y = 4  =>  y = 4 - x
# x - y = 0  =>  y = x
ax = axes[0]
y1 = 4 - x
y2 = x
ax.plot(x, y1, 'b-', linewidth=2, label=r'$x + y = 4$')
ax.plot(x, y2, 'r-', linewidth=2, label=r'$x - y = 0$')
ax.plot(2, 2, 'ko', markersize=8, zorder=5)
ax.annotate('(2, 2)', xy=(2, 2), xytext=(2.5, 1.2),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'))
ax.set_title('Unique Solution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('y')

# --- Case 2: Inconsistent (parallel lines) ---
# x + y = 4  =>  y = 4 - x
# x + y = 1  =>  y = 1 - x
ax = axes[1]
y1 = 4 - x
y2 = 1 - x
ax.plot(x, y1, 'b-', linewidth=2, label=r'$x + y = 4$')
ax.plot(x, y2, 'r-', linewidth=2, label=r'$x + y = 1$')
ax.set_title('Inconsistent (Parallel)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-1, 5)
ax.set_ylim(-2, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('y')

# --- Case 3: Infinitely many solutions (same line) ---
# x + y = 4  =>  y = 4 - x
# 2x + 2y = 8  =>  y = 4 - x  (same line)
ax = axes[2]
y1 = 4 - x
ax.plot(x, y1, 'b-', linewidth=4, alpha=0.5, label=r'$x + y = 4$')
ax.plot(x, y1, 'r--', linewidth=2, label=r'$2x + 2y = 8$')
ax.set_title('Infinitely Many (Coincident)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('y')

plt.tight_layout()
plt.suptitle('Geometry of 2D Linear Systems', fontsize=15, fontweight='bold', y=1.03)
plt.show()

### 5.2 Three dimensions: planes in space

In 3D, each linear equation $ax + by + cz = d$ represents a **plane**.
A system of three equations corresponds to three planes, and the
solution is their common intersection.

| Geometry                         | Algebraic outcome       |
|----------------------------------|-------------------------|
| Three planes meet at a point     | Unique solution         |
| Three planes intersect along a line | Infinitely many solutions |
| Planes have no common point      | Inconsistent            |

In [ ]:
def plot_plane(ax, a, b, c, d, color, alpha=0.25, label=None):
    """Plot the plane ax + by + cz = d over a grid."""
    xx, yy = np.meshgrid(np.linspace(-3, 5, 20), np.linspace(-3, 5, 20))
    if abs(c) > 1e-12:
        zz = (d - a * xx - b * yy) / c
        ax.plot_surface(xx, yy, zz, color=color, alpha=alpha, label=label)
    elif abs(b) > 1e-12:
        zz = np.linspace(-3, 5, 20)
        xx2, zz2 = np.meshgrid(np.linspace(-3, 5, 20), zz)
        yy2 = (d - a * xx2) / b
        ax.plot_surface(xx2, yy2, zz2, color=color, alpha=alpha, label=label)


fig = plt.figure(figsize=(18, 5.5))

# --- Case 1: Three planes meeting at a point ---
ax1 = fig.add_subplot(131, projection='3d')
plot_plane(ax1, 1, 1, 2, 9, 'blue', 0.2)
plot_plane(ax1, 2, 4, -3, 1, 'red', 0.2)
plot_plane(ax1, 3, 6, -5, 0, 'green', 0.2)
# Mark the intersection point (1, 2, 3)
ax1.scatter([1], [2], [3], color='black', s=80, zorder=5)
ax1.set_title('Unique Solution\n(point)', fontsize=12, fontweight='bold')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

# --- Case 2: Three planes intersecting along a line ---
ax2 = fig.add_subplot(132, projection='3d')
plot_plane(ax2, 1, 1, 1, 3, 'blue', 0.2)
plot_plane(ax2, 1, 1, -1, 1, 'red', 0.2)
plot_plane(ax2, 2, 2, 0, 4, 'green', 0.2)
# The common line: x + y = 2, z = 1 => parametrically x=t, y=2-t, z=1
t_line = np.linspace(-2, 4, 50)
ax2.plot(t_line, 2 - t_line, np.ones_like(t_line), 'k-', linewidth=3)
ax2.set_title('Infinitely Many\n(line)', fontsize=12, fontweight='bold')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

# --- Case 3: Parallel planes (no common point) ---
ax3 = fig.add_subplot(133, projection='3d')
plot_plane(ax3, 0, 0, 1, 1, 'blue', 0.2)   # z = 1
plot_plane(ax3, 0, 0, 1, 3, 'red', 0.2)     # z = 3
plot_plane(ax3, 0, 0, 1, 5, 'green', 0.2)   # z = 5
ax3.set_title('Inconsistent\n(parallel planes)', fontsize=12, fontweight='bold')
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')

plt.tight_layout()
plt.suptitle('Geometry of 3D Linear Systems', fontsize=15, fontweight='bold', y=1.02)
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions in the
code cells below.

### Exercise 1 (Easy)

Solve the system:

$$\begin{cases} 2x + y = 5 \\ x - y = 1 \end{cases}$$

**Steps:**
1. Write the augmented matrix as a list of lists.
2. Create a `Matrix` object.
3. Use `solve_system` to find the solution.
4. Verify by plugging the values back into both equations.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Analyze the following system --- determine whether it has a unique solution,
infinitely many solutions, or no solution:

$$\begin{cases}
x + 2y + 3z = 6 \\
2x + 5y + 2z = 4 \\
6x + 15y + z = -2
\end{cases}$$

**Steps:**
1. Encode the augmented matrix.
2. Run `forward_eliminate` with `verbose=True` and study the REF.
3. Run `back_substitute` and interpret the result.
4. If the solution is parametric, substitute a specific value for the free variable(s)
   and verify it satisfies all three equations.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `classify_system(A, b)` that takes a coefficient matrix `A`
(as a list of lists) and a right-hand side vector `b` (as a list), and returns
one of the strings `"unique"`, `"infinite"`, or `"inconsistent"`.

**Constraints:**
- You may use `Matrix.from_lists`, `forward_eliminate`, and direct matrix inspection.
- You should **not** call `back_substitute` or `solve_system`.
- Instead, inspect the row-echelon form to determine the answer:
  - Count the number of pivot columns vs. the number of variables.
  - Check for contradictory rows (all-zero left side, nonzero right side).

Test your function on the three systems from Section 3.9.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept                     | Key takeaway                                                        |
|-----------------------------|---------------------------------------------------------------------|
| Linear equation             | $a_1x_1 + \cdots + a_nx_n = b$; each variable to the first power.  |
| Augmented matrix            | Compact representation $[A \mid \mathbf{b}]$ of a system.          |
| Row operations              | Swap, scale, add-scaled-row --- they preserve the solution set.     |
| Row-echelon form            | Staircase of leading 1s with zeros below.                           |
| Forward elimination         | Systematic reduction to REF using row operations.                   |
| Back-substitution           | Solve from the bottom row upward once in REF.                       |
| Three outcomes              | Unique, infinitely many, or no solution --- no other possibility.   |
| Parametric solutions        | Free variables become parameters; pivot variables depend on them.   |

**Next notebook:** 1.2 --- Gaussian Elimination and Gauss-Jordan Elimination,
where we extend these ideas to **reduced** row-echelon form (RREF) and explore
a more systematic approach to solving larger systems.